In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-core/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 02 · Waiting for days, resuming from anywhere

Human-in-the-loop is the feature that defines long-running agents. The run waits at no cost, and an event wakes the run.

In [2]:
import sys; sys.path[:0] = [".."]                # the lab root, from notebooks/ or solutions/
from core import Engine, Queue, Store, Crash, drain
from workflow import STEPS, CALLS
clock = [0.0]; now = lambda: clock[0]              # a clock we control: days pass in one line
def fresh(steps=STEPS):
    CALLS.clear(); return Engine(Store(), Queue(), dict(steps), clock=now, lease_ttl=60)


## 1. Suspend

The `review` step returns a *wait* outcome. Examine how the run looks during the wait. The run has no task, no lease and nothing to bill.

In [3]:
engine = fresh()
engine.start("run-1", "draft", {"topic": "agents"})
print(drain(engine, engine.queue, now))
run = engine.store.get("run-1")
print(run["status"], run["wait"])
assert run["status"] == "WAITING" and engine.queue.tasks == [] and run["lease"] is None

[('draft', 'ok'), ('review', 'waiting')]
WAITING {'key': 'approval:run-1', 'then': 'publish', 'timeout': 259200.0}


## 2. Resume — two days later, from a different machine

The approval comes as a webhook: `POST /runs/run-1/events {key, payload}`. Fill in the key that the webhook must have.

In [4]:
clock[0] += 2 * 86400
resumed = engine.resume("run-1", "approval:run-1", {"decision": "approve", "by": "editor"})
print("resumed at step:", resumed["step"], "| queued:", engine.queue.tasks)
print(drain(engine, engine.queue, now))
assert engine.store.get("run-1")["result"]["published"] is True

resumed at step: publish | queued: [(172800.0, 'run-1', 'publish', 1)]
[('publish', 'ok'), ('notify', 'succeeded')]


## 3. Duplicates and wrong keys are not errors

A sender can send a webhook two times. A person can click a link two times. What must `resume` return for a second delivery?

In [5]:
engine = fresh()
engine.start("run-2", "draft", {"topic": "x"}); drain(engine, engine.queue, now)
print(engine.resume("run-2", "approval:someone-else", {"decision": "approve"}))
first = engine.resume("run-2", "approval:run-2", {"decision": "approve"})
second = engine.resume("run-2", "approval:run-2", {"decision": "approve"})
assert first is not None and second is None and len(engine.queue.tasks) == 1

None


## 4. Timeouts are absolute

The wait records `timeout = now + 3 days` *at the time of the suspension*. Thus the timeout stays in effect after a restart. The reaper makes sure that the run obeys the timeout. Cloud Scheduler calls `/reap` to start the reaper.

In [6]:
engine = fresh()
engine.start("run-3", "draft", {"topic": "x"}); drain(engine, engine.queue, now)
print("day 1:", engine.reap())
clock[0] += 3 * 86400 + 1
print("later:", engine.reap())
assert engine.store.get("run-3")["status"] == "FAILED"

day 1: []
later: [('timeout', 'run-3')]


## 5. Your turn: a rejection path

Write a `publish_or_not` step. The step reads the decision from `ctx.state["events"][key]`. If the decision is a rejection, the step completes with `{"published": False}`, with no retry and no effect.

In [7]:
def ask(ctx):
    return ("wait", f"ok:{ctx.run_id}", "publish_or_not")
def publish_or_not(ctx):
    decision = ctx.state["events"][f"ok:{ctx.run_id}"]
    if decision["decision"] != "approve":
        return ("done", {"published": False})
    ctx.effect("publish", lambda: CALLS.append(("publish", "x")))
    return ("done", {"published": True})

engine = fresh({"ask": ask, "publish_or_not": publish_or_not})
engine.start("run-4", "ask", {}); drain(engine, engine.queue, now)
engine.resume("run-4", "ok:run-4", {"decision": "reject"}); drain(engine, engine.queue, now)
assert engine.store.get("run-4")["result"] == {"published": False} and CALLS == []
print("rejected cleanly:", engine.store.get("run-4")["history"])

rejected cleanly: [('ask', 1, 'ok', 'wait'), ('publish_or_not', 1, 'ok', 'done')]
